# Pipeline integration checks

Runs all stages on a 17-record synthetic fixture. A randomly initialized tiny Qwen3 model with LoRA exercises real PyTorch forward/backward, hard-negative refresh, checkpointing and scoring without downloading weights. The fixture tokenizer is intentionally synthetic; these checks make no model-quality or real-dataset recall claim. Artifacts stay in a unique artifacts/smoke-* directory.

The tests also verify ownership exclusion, supplemental positives, singleton negatives, loss-gradient equivalence, empty-query retention, threshold isolation and strict submission checks.

In [ ]:
from pathlib import Path
import json
_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else next(
    p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "anshp" / "config.json").is_file())
_A = {"__file__": str(_ROOT / "00_setup_and_data_audit.ipynb")}
for _c in json.loads((_ROOT / "00_setup_and_data_audit.ipynb").read_text(encoding="utf-8"))["cells"]:
    if _c["cell_type"] == "code" and "er-definitions" in _c.get("metadata", {}).get("tags", []):
        exec("".join(_c["source"]), _A)

import copy
import csv
import tempfile
import sqlite3
import hashlib
_C,_=_A["load_helpers"](_ROOT/"01_data_cleaning.ipynb")
_B,_=_A["load_helpers"](_ROOT/"02_blocking.ipynb")
_T,_=_A["load_helpers"](_ROOT/"03_qwen_training.ipynb")
_V,_=_A["load_helpers"](_ROOT/"04_validation_and_thresholds.ipynb")
_S,_=_A["load_helpers"](_ROOT/"05_inference_and_submission.ipynb")

def make_fixture(base):
    data=Path(base)/"data"
    train={
      1:[("S1-fit","Acme LLC","0012 Main Street","US"),
         ("S1-single","MS","7 King Street","US"),
         ("S1-early","Beta Cafe","45 Oak Road","US"),
         ("S1-tune","École Paris","10 Rue Victor","FR"),
         ("S1-hold","Acme Gamma","0012 Main Street","US")],
      2:[("S2-fit","Acme","0012 Main St","US"),
         ("S2-early","Beta Café","45 Oak Rd","US"),
         ("S2-hold","Acme Gamma","0012 Main Street","US"),
         ("S2-unowned","M S","77 King Street","US"),
         ("S2-tune","Ecole Paris","10 Rue Victor","FR")],
      3:[("S3-fit","Hidden Positive","999 Remote Lane","US"),
         ("S3-hold","Gamma Store","8 Lake Road","US"),
         ("S3-noise","Unrelated Service","200 Remote Lane","US")]
    }
    test={
      1:[("S1-test","Acme","0012 Main St","US"),("S1-empty","","","FR")],
      2:[("S2-test","Acme","0012 Main Street","US")],
      3:[("S3-test","No relation","2 Elsewhere","FR")]}
    for dataset,tables in (("train",train),("test",test)):
        (data/dataset).mkdir(parents=True,exist_ok=True)
        for source,rows in tables.items():
            with (data/dataset/f"{dataset}_source{source}.tsv").open("w",encoding="utf-8",newline="") as f:
                w=csv.writer(f,delimiter="\t");w.writerow(_A["SOURCE_COLUMNS"]);w.writerows(rows)
    truth={"S1-fit":["S2-fit","S3-fit"],"S1-single":[],"S1-early":["S2-early"],
           "S1-tune":["S2-tune"],"S1-hold":["S2-hold","S3-hold"]}
    with (data/"train/train_ground_truth.tsv").open("w",encoding="utf-8",newline="") as f:
        w=csv.writer(f,delimiter="\t");w.writerow(["source1_entity_id","matched_entity_ids"])
        w.writerows((q,",".join(c)) for q,c in truth.items())
    splits={"S1-fit":"fit","S1-single":"fit","S1-early":"early_stop",
            "S1-tune":"threshold_tune","S1-hold":"holdout"}
    return data,truth,splits

class FixtureTokenizer:
    """Whitespace/hash tokenizer for wiring tests only; never a Qwen tokenizer."""
    pad_token_id=0
    padding_side="left"
    def encode(self,text,add_special_tokens=False):
        if text=="yes":return [9]
        if text=="no":return [10]
        return [11+int(hashlib.sha256(t.encode("utf-8")).hexdigest()[:8],16)%117
                for t in text.split()]
    def convert_tokens_to_ids(self,text):
        return {"yes":9,"no":10}[text]
    def pad(self,items,padding=True,return_tensors="pt"):
        import torch
        width=max(len(x["input_ids"]) for x in items)
        return {"input_ids":torch.tensor([[0]*(width-len(x["input_ids"]))+x["input_ids"] for x in items]),
                "attention_mask":torch.tensor([[0]*(width-len(x["input_ids"]))+x["attention_mask"] for x in items])}
    def save_pretrained(self,path):
        path=Path(path);path.mkdir(parents=True,exist_ok=True)
        _A["atomic_json"](path/"FIXTURE_ONLY.json",{"synthetic":True})

def run_pipeline_checks():
    import torch
    from transformers import Qwen3Config,Qwen3ForCausalLM
    from peft import LoraConfig,TaskType,get_peft_model
    base=Path(tempfile.mkdtemp(prefix="smoke-",dir=_ROOT/"artifacts"))
    data,truth,splits=make_fixture(base)
    cfg=json.loads((_ROOT/"config.json").read_text(encoding="utf-8"))
    cfg["synthetic_fixture"]=True
    cfg["training"].update(max_epochs=2,micro_batch_size=2,reference_groups_per_step=2)
    audit=_A["build_data_audit"](data,base/"audit",cfg,splits)
    assert tuple(audit["truth_summary"])==(5,1,6,2)
    assert _A["build_data_audit"](data,base/"audit",cfg,splits)["fingerprint"]==audit["fingerprint"]
    norm=_C["build_normalized"](base/"audit",base/"normalized",batch_size=2)
    assert sum(x["rows"] for x in norm["files"])==17
    for dataset in ("train","test"):
        _B["build_retrieval_index"](base/"normalized",base/"audit",base/("index_"+dataset),dataset)
    for split in _A["SPLIT_NAMES"]:
        _B["build_candidates"](base/"index_train",base/"audit",base/("candidates_"+split),
                               cap=4,channel_limit=8,split=split)
    _B["build_candidates"](base/"index_test",base/"audit",base/"candidates_test",cap=4,channel_limit=8)
    fm=_T["prepare_fit_data"](base/"candidates_fit",base/"fit",cfg)
    con=_B["sqlite_readonly"](base/"fit"/fm["database"])
    examples=con.execute("SELECT * FROM examples").fetchall()
    assert all(r[1] not in {"S2-hold","S3-hold","S2-early","S2-tune"} for r in examples)
    assert any(r[1]=="S3-fit" and r[3]=="supplemental_positive" for r in examples)
    assert any(r[0]=="S1-single" and r[2]==0 for r in examples)
    con.close()

    # Verify accumulation gradients against the complete reference-normalized objective.
    z=torch.tensor([.4,-.2,.6],requires_grad=True)
    labels=[1,0,1];weights=torch.tensor([.5,.5,1.])
    loss=(_T["qwen_independent_pair_loss"](z,labels)*weights).sum()/2
    expected=torch.autograd.grad(loss,z)[0]
    z2=z.detach().clone().requires_grad_()
    for i in range(3):
        (_T["qwen_independent_pair_loss"](z2[i:i+1],labels[i:i+1])*weights[i]/2).sum().backward()
    assert torch.allclose(expected,z2.grad)

    torch.manual_seed(2026)
    tiny=Qwen3ForCausalLM(Qwen3Config(vocab_size=128,hidden_size=32,intermediate_size=64,
        num_hidden_layers=1,num_attention_heads=4,num_key_value_heads=2,head_dim=8,
        max_position_embeddings=1024,pad_token_id=0,eos_token_id=1))
    model=get_peft_model(tiny,LoraConfig(task_type=TaskType.CAUSAL_LM,r=2,lora_alpha=4,
                         target_modules=["q_proj","v_proj"],lora_dropout=0.0))
    tok=FixtureTokenizer()
    model.eval()
    pairs=[({"business_name":"A","business_address":"12 Road","country":"US"},
            {"business_name":"A Company","business_address":"12 Long Road","country":"US"}),
           ({"business_name":"B","business_address":"","country":"FR"},
            {"business_name":"B","business_address":"","country":"FR"})]
    batch,_=_T["pair_batch"](tok,pairs,"cpu",256)
    with torch.inference_mode():
        full=model(**batch,use_cache=False).logits
        expected_scores=_T["qwen_pair_log_odds"](full,batch["attention_mask"],9,10)
        assert torch.allclose(expected_scores,_T["forward_pair_scores"](model,batch,9,10),atol=1e-6)
    before={k:p.detach().clone() for k,p in model.named_parameters() if p.requires_grad}
    def early(model,epoch):
        path=base/f"scores_early_{epoch}"
        _T["score_candidates"](model,tok,(9,10),base/"candidates_early_stop",path,cfg,
                              f"synthetic-epoch-{epoch}")
        return _V["evaluate_scored_set"](path,"early_stop")["metrics"]
    trained=_T["train_qwen"](model,tok,(9,10),base/"fit",base/"model",cfg,early)
    assert any(not torch.equal(before[k],p.detach()) for k,p in model.named_parameters() if k in before)
    # Use one unchanged model identity for tune/holdout/test; these scores are synthetic.
    for split in ("threshold_tune","holdout","test"):
        _T["score_candidates"](model,tok,(9,10),base/("candidates_"+split),
                              base/("scores_"+split),cfg,"SYNTHETIC-FIXTURE")
    # Completed-run reuse is deterministic.
    reused=_T["score_candidates"](model,tok,(9,10),base/"candidates_test",
                                 base/"scores_test",cfg,"SYNTHETIC-FIXTURE")
    score_path=base/"scores_test"/reused["database"]
    with sqlite3.connect(score_path) as score_db:
        score_db.execute("DELETE FROM scores WHERE reference_id='S1-test'")
    # This file is inside the newly created fixture directory; emulate an interrupted run.
    (base/"scores_test/manifest.json").unlink()
    resumed=_T["score_candidates"](model,tok,(9,10),base/"candidates_test",
                                  base/"scores_test",cfg,"SYNTHETIC-FIXTURE")
    assert resumed["profile"]["new_pairs"]==1
    _V["tune_threshold"](base/"scores_threshold_tune",base/"policy")
    _V["evaluate_holdout"](base/"scores_holdout",base/"policy",base/"holdout.json")
    result=_S["export_submission"](base/"scores_test",base/"policy",base/"output")
    assert result["validation"]["references"]==2
    assert result["validation"]["candidate_pairs"]==1
    with (base/"output/candidate_pairs.tsv").open(encoding="utf-8",newline="") as f:
        rows=list(csv.reader(f,delimiter="\t"))
    assert ["S1-empty",""] in rows

    # Corrupted exported candidate sets must be rejected, including missing empty rows.
    bad=base/"bad.tsv"
    bad.write_text("source1_entity_id\tcandidate_entity_ids\nS1-test\tS2-test\n",encoding="utf-8")
    try:
        _S["validate_submission"](base/"output/matching_results.tsv",bad,base/"candidates_test")
    except ValueError:pass
    else:raise AssertionError("Missing reference accepted")
    try:
        _V["evaluate_scored_set"](base/"scores_holdout","threshold_tune")
    except ValueError:pass
    else:raise AssertionError("Holdout entered threshold tuning")

    # Country hard blocking is admitted only after a full true-link consistency audit.
    changed=data/"train/train_source2.tsv"
    changed.write_text(changed.read_text(encoding="utf-8").replace(
        "S2-fit\tAcme\t0012 Main St\tUS","S2-fit\tAcme\t0012 Main St\tFR"),encoding="utf-8")
    try:_A["build_data_audit"](data,base/"bad_audit",cfg,splits)
    except ValueError as e:assert "Country partition" in str(e)
    else:raise AssertionError("Cross-country gold link accepted")

    report={"passed":True,"synthetic_only":True,"fixture_directory":str(base),
            "checks":["audit","normalization","country_gate","blocking","empty_scope",
                      "ownership_exclusion","supplemental_positive","singleton_negative",
                      "reference_gradient_equivalence","tiny_qwen_lora_updates",
                      "last_token_score_parity","hard_negative_refresh","complete_candidate_scoring","score_reuse","partial_score_resume",
                      "threshold_isolation","locked_holdout","submission_validation"],
            "submission":result["validation"],"environment":_A["environment_snapshot"]()}
    _A["atomic_json"](_ROOT/"reports"/"pipeline_smoke.json",report)
    print(json.dumps(report,indent=2))
    return report


In [ ]:
smoke_report = run_pipeline_checks()
